###Author - Haaris Khalil
###Course - Data Management

###Week 2- Exercise – Individual - Data Quality Audit

### 1. Dataset Description & Business Use Case
This dataset contains 11 records of customer financial transactions, tracking elements like customer IDs, transaction dates, payment methods, currency, region, and transaction status.

**Business Use Case:** The primary use case for this data is likely revenue reporting, customer behavior analytics, and payment processing audits. High data quality is critical here, because incorrect amounts, missing IDs, or mismatched currencies could severely distort financial reports and lead to incorrect business decisions.

In [1]:
import pandas as pd
import numpy as np

# Load the dataset (Make sure the CSV is uploaded to your Colab session)
df = pd.read_csv('week2_customer_transactions_messy.csv')

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

Dataset loaded successfully.
Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


### 2. Issues Mapped to Data Quality Dimensions
Based on an initial inspection of the raw data, here are the quality issues categorized by standard dimensions:

* **Completeness:** Missing values exist in `customer_id` (T0004), `amount` (T0009), `payment_method` (T0010), and `last_updated`.
* **Uniqueness:** There is an exact duplicate row for transaction T0006.
* **Validity:** Amounts of 0 and negative 35.00 are invalid for standard completed sales. The date `2026-02-30` is impossible.
* **Consistency:** Date formats are highly mixed (YYYY/MM/DD, DD-MM-YYYY, YYYY-MM-DD). Text capitalization is inconsistent (EUR vs EURO, DE vs de, card vs CARD).
* **Integrity:** Transaction T0007 has a `transaction_date` (Feb 30) that logically occurs after its `last_updated` date (Feb 15), violating time sequence integrity.

In [3]:
#@title 3. Compute 3 KPIs

total_rows = len(df)
total_cells = df.shape[0] * df.shape[1]

# KPI 1: Completeness Rate (percentage of non-null cells)
completeness_rate = df.notna().sum().sum() / total_cells * 100

# KPI 2: Duplication Rate (percentage of duplicate rows)
duplication_rate = df.duplicated().sum() / total_rows * 100

# KPI 3: Valid Amount Rate (percentage of rows with amounts > 0)
valid_amount_rate = (df['amount'] > 0).sum() / total_rows * 100

print(f"Completeness Rate (Cell Level): {completeness_rate:.1f}%")
print(f"Duplication Rate (Row Level): {duplication_rate:.1f}%")
print(f"Valid Amount Rate (>0): {valid_amount_rate:.1f}%")

Completeness Rate (Cell Level): 96.0%
Duplication Rate (Row Level): 9.1%
Valid Amount Rate (>0): 72.7%


In [4]:
#@title 4. Define Validation Rules & Audit Summary

# Dictionary of validation rules (True indicates a rule violation)
rules = {
    "missing_data_any_column": df.isnull().any(axis=1),
    "exact_row_duplicates": df.duplicated(keep='first'),
    "invalid_or_negative_amounts": (df['amount'] <= 0) | df['amount'].isnull(),
    "inconsistent_text_formatting": df['currency'].isin(['EURO']) | df['region'].str.islower().fillna(False)
}

def summarize_rule_violations(rule_dictionary):
    """
    Table with one row per rule and violation counts.
    """
    return pd.DataFrame({
        'rule_name': list(rule_dictionary.keys()),
        'affected_rows': [int(mask.sum()) for mask in rule_dictionary.values()]
    }).sort_values('affected_rows', ascending=False)

audit_df = summarize_rule_violations(rules)

print("Audit Summary Results:")
display(audit_df)

Audit Summary Results:


,rule_name,affected_rows
0,missing_data_any_column,3
2,invalid_or_negative_amounts,3
3,inconsistent_text_formatting,2
1,exact_row_duplicates,1


### Explain your function parameters
The `rule_dictionary` parameter accepts a Python dictionary where the keys are descriptive rule names (strings) and the values are boolean arrays (pandas Series) flagging which rows violate that specific rule.

Default values (like checking if `amount <= 0`) were selected based on standard e-commerce logic to capture fundamental errors without flagging legitimate edge cases. Changing these parameters to be stricter (for example, flagging amounts over 500 as anomalies) would increase the number of affected rows, thereby lowering the valid amount KPI and requiring more manual data review.

### Recommended Cleaning Actions
1. **Deduplication:** Apply `df.drop_duplicates()` to remove identical rows safely.
2. **Standardization:** Apply `.str.upper()` and `.str.strip()` to the `currency`, `payment_method`, and `region` columns to resolve case and spacing inconsistencies.
3. **Date Parsing:** Coerce all date columns to standard datetime objects using `pd.to_datetime(..., errors='coerce')` so that completely invalid dates (like Feb 30) become `NaT` (Not a Time), making them easier to filter or impute.
4. **Handling Outliers:** Filter out rows where the amount is less than or equal to zero, or flag them in a separate `is_refund` column if negative amounts represent customer returns.